In [ ]:
import os
import re
import csv
import time
import random
import shutil
import tempfile
import traceback

from math import ceil
from dataclasses import dataclass
from datetime import date, timedelta
from typing import List, Tuple, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import (
    TimeoutException,
    WebDriverException,
    NoAlertPresentException,
    StaleElementReferenceException,
    UnexpectedAlertPresentException,
    ElementClickInterceptedException,
)

# =========================
# Types
# =========================
DateTuple = Tuple[int, int, int]
RangeTuple = Tuple[DateTuple, DateTuple]

# =========================
# Config
# =========================
@dataclass
class CrawlConfig:
    base_url: str = "https://www.infocare.co.kr/"
    userid: str = "광주은행"
    passwd: str = "1234"

    wait_sec: int = 25
    hold_browser: bool = False           # 전체 크롤링이면 보통 False 권장
    hold_browser_on_error: bool = True
    cleanup_profile_dir_on_exit: bool = True

    min_delay: float = 0.3
    max_delay: float = 1.2

    region: str = "광주"
    rows_per_page: str = "50"           # 50 고정

    # ✅ 이번 실행은 "최근 4주(-2주~+2주)"만 저장할 파일
    output_csv: str = "data/gwangju_recent_4w.csv"

    # ✅ 오늘 기준 ±14일
    window_days: int = 60


# =========================
# Logging / timing
# =========================
def log(msg: str):
    print(f"[LOG] {msg}")

def jitter_sleep(cfg: CrawlConfig):
    time.sleep(random.uniform(cfg.min_delay, cfg.max_delay))


# =========================
# Robust helpers
# =========================
def handle_unexpected_alert(driver, accept=True, timeout=2) -> bool:
    try:
        WebDriverWait(driver, timeout).until(EC.alert_is_present())
        alert = driver.switch_to.alert
        text = ""
        try:
            text = alert.text
        except Exception:
            pass

        if accept:
            alert.accept()
            log(f"예상치 못한 alert 처리: accept (text='{text}')")
        else:
            alert.dismiss()
            log(f"예상치 못한 alert 처리: dismiss (text='{text}')")
        return True

    except (TimeoutException, NoAlertPresentException):
        return False
    except Exception as e:
        log(f"alert 처리 중 예외(무시 가능): {e}")
        return False


def safe_click(driver, wait, by, selector, desc, retries=2):
    last_err = None
    for attempt in range(1, retries + 2):
        try:
            el = wait.until(EC.element_to_be_clickable((by, selector)))
            el.click()
            log(f"{desc} 클릭 완료")
            return el

        except UnexpectedAlertPresentException as e:
            last_err = e
            handle_unexpected_alert(driver, accept=True, timeout=3)
            log(f"{desc}: alert 때문에 재시도({attempt}/{retries+1})")

        except (ElementClickInterceptedException, StaleElementReferenceException, WebDriverException) as e:
            last_err = e
            handle_unexpected_alert(driver, accept=True, timeout=1)
            log(f"{desc}: 클릭 예외로 재시도({attempt}/{retries+1}) - {type(e).__name__}")

        except TimeoutException as e:
            last_err = e
            handle_unexpected_alert(driver, accept=True, timeout=1)
            log(f"{desc}: 대기 timeout 재시도({attempt}/{retries+1})")

    raise last_err


def safe_select_by_value(driver, wait, by, selector, value, desc, retries=2):
    """
    일반 select 유틸(다른 곳에서 사용)
    - rows-per-page는 별도 함수(set_rows_per_page)에서 더 강하게 처리함
    """
    last_err = None
    for attempt in range(1, retries + 2):
        try:
            el = wait.until(EC.presence_of_element_located((by, selector)))
            Select(el).select_by_value(value)
            log(f"{desc} 선택 완료: {value}")
            return

        except UnexpectedAlertPresentException as e:
            last_err = e
            handle_unexpected_alert(driver, accept=True, timeout=3)
            log(f"{desc}: alert 때문에 재시도({attempt}/{retries+1})")

        except (StaleElementReferenceException, WebDriverException, TimeoutException) as e:
            last_err = e
            handle_unexpected_alert(driver, accept=True, timeout=1)
            log(f"{desc}: 선택 예외로 재시도({attempt}/{retries+1}) - {type(e).__name__}")

    raise last_err


def switch_to_info_main_if_exists(driver, wait) -> bool:
    driver.switch_to.default_content()
    if driver.find_elements(By.NAME, "info_main"):
        wait.until(EC.frame_to_be_available_and_switch_to_it((By.NAME, "info_main")))
        return True
    return False


def ensure_info_main(driver, wait) -> bool:
    in_frame = switch_to_info_main_if_exists(driver, wait)
    if not in_frame:
        driver.switch_to.default_content()
    return in_frame


def clear_site_session(driver):
    try:
        driver.switch_to.default_content()
    except Exception:
        pass

    try:
        driver.delete_all_cookies()
        log("쿠키 삭제 완료")
    except Exception as e:
        log(f"쿠키 삭제 실패(무시 가능): {e}")

    try:
        driver.execute_script("window.localStorage.clear();")
        driver.execute_script("window.sessionStorage.clear();")
        log("localStorage / sessionStorage 삭제 완료")
    except Exception as e:
        log(f"스토리지 삭제 실패(무시 가능): {e}")

    try:
        driver.execute_cdp_cmd("Network.clearBrowserCache", {})
        log("브라우저 캐시 삭제 완료(CDP)")
    except Exception as e:
        log(f"캐시 삭제 실패(무시 가능): {e}")


# =========================
# Date range: today ± window_days
# =========================
def date_to_tuple(d: date) -> DateTuple:
    return (d.year, d.month, d.day)

def get_today_window_range(window_days: int, today: Optional[date] = None) -> RangeTuple:
    if today is None:
        today = date.today()
    start_d = today - timedelta(days=window_days)
    end_d = today + timedelta(days=window_days)
    return (date_to_tuple(start_d), date_to_tuple(end_d))


# =========================
# Driver / navigation
# =========================
def build_driver(cfg: CrawlConfig):
    profile_dir = tempfile.mkdtemp(prefix="selenium-infocare-")

    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-gpu")
    options.add_argument(f"--user-data-dir={profile_dir}")

    driver = webdriver.Chrome(options=options)
    try:
        # Chrome 실행 직후 캐시를 1회 삭제
        driver.execute_cdp_cmd("Network.enable", {})
        driver.execute_cdp_cmd("Network.clearBrowserCache", {})
        log("브라우저 캐시 초기화 완료")
    except Exception as e:
        log(f"브라우저 캐시 초기화 실패(무시): {e}")
    wait = WebDriverWait(driver, cfg.wait_sec)
    return driver, wait, profile_dir


def login_and_go_to_total_search(driver, wait, cfg: CrawlConfig):
    driver.get(cfg.base_url)
    log("메인 접속 완료")

    ensure_info_main(driver, wait)
    log("info_main 프레임 진입 완료(초기)")

    try:
        safe_click(driver, wait, By.CSS_SELECTOR, "ul.hd_login li.login a", "로그인 메뉴")
    except Exception:
        log("로그인 메뉴 없음/이미 로그인 화면일 수 있어 스킵")

    wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "div.pop-up-background.login-pane")))
    userid = wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "div.pop-up-background.login-pane form.login input.userid")
    ))
    passwd = wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "div.pop-up-background.login-pane form.login input.passwd")
    ))

    userid.clear()
    userid.send_keys(cfg.userid)
    passwd.clear()
    passwd.send_keys(cfg.passwd)
    log("ID/PW 입력 완료")

    login_btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//div[contains(@class,'login-pane') and contains(@class,'pop-up-background')]//button[normalize-space()='로그인']")
    ))
    ActionChains(driver).move_to_element(login_btn).pause(0.1).click(login_btn).perform()
    log("로그인 버튼 클릭 완료")

    handle_unexpected_alert(driver, accept=True, timeout=2)

    ensure_info_main(driver, wait)
    log("로그인 후 info_main 프레임 진입 완료")

    safe_click(driver, wait, By.CSS_SELECTOR, "li.main_nav_li02 > a", "법원경매")
    handle_unexpected_alert(driver, accept=True, timeout=2)

    safe_click(driver, wait, By.CSS_SELECTOR, "a[href='/bubwon/search/search_total.asp']", "통합검색")

    driver.switch_to.default_content()
    has_frame = bool(driver.find_elements(By.NAME, "info_main"))
    log(f"통합검색 클릭 후 info_main 프레임 존재 여부: {has_frame}")

    ensure_info_main(driver, wait)
    log("✅ 통합검색 진입 완료: 여기부터 크롤링 시작")


# =========================
# Result parsing & pagination
# =========================
TABLE_SELECTOR = "table.sub_table_wr.area_table_wr.table_list_wr.mulgun-list"
TBODY_SELECTOR = "tbody.mulgun-list"
ROWS_PER_PAGE_SELECTOR = "select[name='number'].rows-per-page"
PAGINATION_UL_SELECTOR = "ul.clearfix.pagenation"
TOTAL_COUNT_SELECTOR = "span.total-count"

CSV_COLUMNS = ["사건번호", "용도", "소재지", "감정가", "최저가", "결과", "낙찰가", "낙찰율", "매각일"]


def wait_for_results_table(driver, wait):
    ensure_info_main(driver, wait)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, TABLE_SELECTOR)))
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, TBODY_SELECTOR)))


def set_rows_per_page(driver, wait, cfg: CrawlConfig):
    """
    ✅ 강화버전: ElementNotInteractable 방지
    - scrollIntoView 후 Select 시도
    - 실패하면 JS로 value 세팅 + change/input 이벤트 강제
    - total-count=0이면 호출하지 않는 것이 원칙(아래 crawl 함수에서 처리)
    """
    ensure_info_main(driver, wait)

    el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ROWS_PER_PAGE_SELECTOR)))

    # 1) 화면에 보이도록 스크롤
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center', inline:'nearest'});", el)
        time.sleep(0.2)
    except Exception:
        pass

    # 2) 일반 Select 시도
    try:
        Select(el).select_by_value(cfg.rows_per_page)
        log(f"페이지당 건수(rows-per-page) 선택 완료: {cfg.rows_per_page}")
    except Exception as e:
        log(f"Select로 rows-per-page 변경 실패 → JS로 재시도: {type(e).__name__}")

        # 3) JS fallback
        driver.execute_script(
            """
            const sel = arguments[0];
            const val = arguments[1];
            sel.value = val;
            sel.dispatchEvent(new Event('change', {bubbles:true}));
            sel.dispatchEvent(new Event('input', {bubbles:true}));
            """,
            el,
            cfg.rows_per_page
        )
        log(f"JS로 rows-per-page 변경 완료: {cfg.rows_per_page}")

    handle_unexpected_alert(driver, accept=True, timeout=1)

    # 변경 후 재로딩 대기
    wait_for_results_table(driver, wait)
    jitter_sleep(cfg)


def get_total_count(driver, wait) -> int:
    ensure_info_main(driver, wait)
    el = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, TOTAL_COUNT_SELECTOR)))
    txt = (el.text or "").strip().replace(",", "")
    return int(txt) if txt.isdigit() else 0


def get_current_on_page(driver, wait) -> int:
    ensure_info_main(driver, wait)
    ul = driver.find_element(By.CSS_SELECTOR, PAGINATION_UL_SELECTOR)
    on = ul.find_element(By.CSS_SELECTOR, "li.on")
    t = (on.text or "").strip()
    return int(t) if t.isdigit() else 1


def get_visible_page_numbers(driver, wait) -> List[int]:
    ensure_info_main(driver, wait)
    ul = driver.find_element(By.CSS_SELECTOR, PAGINATION_UL_SELECTOR)
    lis = ul.find_elements(By.CSS_SELECTOR, "li[data-page]")
    out = []
    for li in lis:
        dp = li.get_attribute("data-page")
        if dp and dp.isdigit():
            out.append(int(dp))
    return sorted(set(out))


def goto_page(driver, wait, target_page: int, cfg: CrawlConfig):
    current = get_current_on_page(driver, wait)
    if current == target_page:
        return

    while True:
        visible = get_visible_page_numbers(driver, wait)
        vmin, vmax = (min(visible), max(visible)) if visible else (target_page, target_page)

        if vmin <= target_page <= vmax:
            ensure_info_main(driver, wait)
            tbody = driver.find_element(By.CSS_SELECTOR, TBODY_SELECTOR)
            trs = tbody.find_elements(By.CSS_SELECTOR, "tr")
            first_tr = trs[0] if trs else None

            safe_click(
                driver, wait,
                By.CSS_SELECTOR,
                f"{PAGINATION_UL_SELECTOR} li[data-page='{target_page}']",
                f"페이지 이동({target_page})"
            )
            handle_unexpected_alert(driver, accept=True, timeout=1)

            try:
                if first_tr is not None:
                    WebDriverWait(driver, 10).until(EC.staleness_of(first_tr))
                else:
                    WebDriverWait(driver, 10).until(lambda d: get_current_on_page(driver, wait) == target_page)
            except Exception:
                WebDriverWait(driver, 10).until(lambda d: get_current_on_page(driver, wait) == target_page)

            wait_for_results_table(driver, wait)
            jitter_sleep(cfg)
            return

        if target_page > vmax:
            safe_click(driver, wait, By.CSS_SELECTOR, f"{PAGINATION_UL_SELECTOR} li.nextpg", "페이지 그룹 다음(nextpg)")
            handle_unexpected_alert(driver, accept=True, timeout=1)
            wait_for_results_table(driver, wait)
            jitter_sleep(cfg)
            continue

        if target_page < vmin:
            safe_click(driver, wait, By.CSS_SELECTOR, f"{PAGINATION_UL_SELECTOR} li.prevpg", "페이지 그룹 이전(prevpg)")
            handle_unexpected_alert(driver, accept=True, timeout=1)
            wait_for_results_table(driver, wait)
            jitter_sleep(cfg)
            continue


def normalize_address(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s)
    return s


def parse_bid_info_from_title(title_text: str):
    if not title_text:
        return "0", "0%"

    m_price = re.search(r"낙찰가:\s*([0-9,]+)원", title_text)
    낙찰가 = m_price.group(1) if m_price else "0"

    m_rate = re.search(r"\(\s*([0-9.]+%)\s*\)", title_text)
    낙찰율 = m_rate.group(1) if m_rate else "0%"

    return 낙찰가, 낙찰율


def parse_one_row(tr):
    tds = tr.find_elements(By.CSS_SELECTOR, "td")
    if len(tds) < 7:
        return None

    # 사건번호: "광주 7계 2023-9687(5)"
    case_text = tds[1].text.strip()
    case_lines = [x.strip() for x in case_text.split("\n") if x.strip()]
    if len(case_lines) >= 2:
        사건번호 = f"{case_lines[0]} {case_lines[1]}"
    elif len(case_lines) == 1:
        사건번호 = case_lines[0]
    else:
        사건번호 = ""

    용도 = tds[2].text.strip()

    addr = ""
    try:
        divs = tds[3].find_elements(By.CSS_SELECTOR, "div")
        if divs:
            addr = divs[-1].text.strip()
    except Exception:
        addr = tds[3].text.strip()
    소재지 = normalize_address(addr)

    감정가, 최저가 = "0", "0"
    try:
        lis = tds[4].find_elements(By.CSS_SELECTOR, "ul li")
        if len(lis) >= 2:
            감정가 = lis[0].text.strip()
            최저가 = lis[1].text.strip()
    except Exception:
        pass

    결과 = tds[5].text.strip()
    매각일 = tds[6].text.strip()

    title_text = tr.get_attribute("title") or ""
    낙찰가, 낙찰율 = parse_bid_info_from_title(title_text)

    if ("낙찰" not in 결과) or (not title_text.strip()):
        낙찰가, 낙찰율 = "0", "0%"

    return {
        "사건번호": 사건번호,
        "용도": 용도,
        "소재지": 소재지,
        "감정가": 감정가,
        "최저가": 최저가,
        "결과": 결과,
        "낙찰가": 낙찰가,
        "낙찰율": 낙찰율,
        "매각일": 매각일,
    }


def collect_current_page_rows(driver, wait) -> List[dict]:
    ensure_info_main(driver, wait)
    tbody = driver.find_element(By.CSS_SELECTOR, TBODY_SELECTOR)
    trs = tbody.find_elements(By.CSS_SELECTOR, "tr")
    out = []
    for tr in trs:
        item = parse_one_row(tr)
        if item:
            out.append(item)
    return out


def crawl_current_result_pages(driver, wait, cfg: CrawlConfig) -> List[dict]:
    """
    - total-count로 총 페이지 계산
    - total-count==0이면 바로 return []
    - rows-per-page는 강화버전으로 50 세팅
    - 1..total_pages 순회하며 페이지 이동/수집
    """
    wait_for_results_table(driver, wait)

    total_count = get_total_count(driver, wait)
    if total_count <= 0:
        log("total-count=0 → 결과 없음, 페이지 크롤링 스킵")
        return []

    # ✅ total_count가 있을 때만 rows-per-page 변경
    set_rows_per_page(driver, wait, cfg)

    per_page = int(cfg.rows_per_page)
    total_pages = ceil(total_count / per_page) if total_count > 0 else 1
    log(f"총 건수(total-count)={total_count}, 페이지당={per_page}, 총 페이지={total_pages}")

    all_rows: List[dict] = []
    for p in range(1, total_pages + 1):
        if p > 1:
            goto_page(driver, wait, p, cfg)

        page_no = get_current_on_page(driver, wait)
        rows = collect_current_page_rows(driver, wait)
        all_rows.extend(rows)
        log(f"페이지 {page_no}/{total_pages} 수집: {len(rows)}건, 누적={len(all_rows)}")

        jitter_sleep(cfg)

    return all_rows


def set_search_filters_and_search(driver, wait, cfg: CrawlConfig, region: str, start: DateTuple, end: DateTuple):
    ensure_info_main(driver, wait)

    sy, sm, sd = start
    ey, em, ed = end

    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='addr_do']", region, "지역(addr_do)")
    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='startyear']", f"{sy}", "시작년도(startyear)")
    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='startmonth']", f"{sm:02d}", "시작월(startmonth)")
    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='startdate']", f"{sd:02d}", "시작일(startdate)")

    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='endyear']", f"{ey}", "종료년도(endyear)")
    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='endmonth']", f"{em:02d}", "종료월(endmonth)")
    safe_select_by_value(driver, wait, By.CSS_SELECTOR, "select[name='enddate']", f"{ed:02d}", "종료일(enddate)")

    safe_click(driver, wait, By.CSS_SELECTOR, "div.cont_btn_wr.button-pane a.search", "검색하기")
    handle_unexpected_alert(driver, accept=True, timeout=2)

    wait_for_results_table(driver, wait)
    jitter_sleep(cfg)


def write_rows_to_csv(csv_path: str, rows: List[dict]):
    """
    ✅ 이번 모드는 '업데이트용 4주 데이터'만 별도 파일로 저장하므로 덮어쓰기(write)
    """
    with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in CSV_COLUMNS})


# =========================
# Main
# =========================
def main():
    cfg = CrawlConfig()

    driver = None
    profile_dir = None

    try:
        driver, wait, profile_dir = build_driver(cfg)

        # 1) 로그인 + 통합검색 진입
        login_and_go_to_total_search(driver, wait, cfg)

        # 2) ✅ 오늘 기준 -2주 ~ +2주 구간 1개만 실행
        today = date.today()
        (start, end) = get_today_window_range(cfg.window_days, today=today)
        log(f"✅ UPDATE MODE: 오늘({today}) 기준 ±{cfg.window_days}일 구간만 실행")
        log(f"=== 검색 구간: {start} ~ {end} ===")

        # 3) 검색 → 모든 페이지 크롤링
        set_search_filters_and_search(driver, wait, cfg, cfg.region, start, end)

        rows = crawl_current_result_pages(driver, wait, cfg)
        log(f"구간 완료: 수집 row={len(rows)}")

        # 4) ✅ CSV로 저장(덮어쓰기)
        write_rows_to_csv(cfg.output_csv, rows)
        log(f"CSV 저장 완료(write): {cfg.output_csv}")

        if cfg.hold_browser:
            log("정상 흐름 완료. 브라우저 유지합니다. (엔터 치면 세션 정리 후 종료)")
            input()

    except Exception as e:
        log(f"❗에러 발생: {type(e).__name__}: {e}")
        traceback.print_exc()

        if driver is not None and cfg.hold_browser_on_error:
            handle_unexpected_alert(driver, accept=True, timeout=2)
            log("에러 발생했지만 브라우저 유지합니다. (상태 확인 후 엔터 치면 종료)")
            input()

    finally:
        if driver is not None:
            try:
                clear_site_session(driver)
            except Exception:
                pass

            try:
                driver.quit()
                log("드라이버 종료")
            except Exception:
                pass

        if cfg.cleanup_profile_dir_on_exit and profile_dir:
            try:
                shutil.rmtree(profile_dir, ignore_errors=True)
                log(f"Selenium 프로필 폴더 삭제 완료: {profile_dir}")
            except Exception as e:
                log(f"프로필 폴더 삭제 실패(무시 가능): {e}")


if __name__ == "__main__":
    main()

[LOG] 브라우저 캐시 초기화 완료
[LOG] 메인 접속 완료
[LOG] info_main 프레임 진입 완료(초기)
[LOG] 로그인 메뉴 클릭 완료
[LOG] ID/PW 입력 완료
[LOG] 로그인 버튼 클릭 완료
[LOG] 로그인 후 info_main 프레임 진입 완료
[LOG] 법원경매 클릭 완료
[LOG] 통합검색 클릭 완료
[LOG] 통합검색 클릭 후 info_main 프레임 존재 여부: True
[LOG] ✅ 통합검색 진입 완료: 여기부터 크롤링 시작
[LOG] ✅ UPDATE MODE: 오늘(2026-03-06) 기준 ±60일 구간만 실행
[LOG] === 검색 구간: (2026, 1, 5) ~ (2026, 5, 5) ===
[LOG] 지역(addr_do): 선택 예외로 재시도(1/3) - ElementNotInteractableException
[LOG] 지역(addr_do) 선택 완료: 광주
[LOG] 시작년도(startyear) 선택 완료: 2026
[LOG] 시작월(startmonth) 선택 완료: 01
[LOG] 시작일(startdate) 선택 완료: 05
[LOG] 종료년도(endyear) 선택 완료: 2026
[LOG] 종료월(endmonth) 선택 완료: 05
[LOG] 종료일(enddate) 선택 완료: 05
[LOG] 검색하기 클릭 완료
[LOG] 페이지당 건수(rows-per-page) 선택 완료: 50
[LOG] 총 건수(total-count)=986, 페이지당=50, 총 페이지=20
[LOG] 페이지 1/20 수집: 50건, 누적=50
[LOG] 페이지 이동(2) 클릭 완료
[LOG] 페이지 2/20 수집: 50건, 누적=100
[LOG] 페이지 이동(3) 클릭 완료
[LOG] 페이지 3/20 수집: 50건, 누적=150
[LOG] 페이지 이동(4) 클릭 완료
[LOG] 페이지 4/20 수집: 50건, 누적=200
[LOG] 페이지 이동(5) 클릭 완료
[LOG] 페이지 5/20 수집: 50건, 누적=250
[LOG] 페이

In [11]:
import pandas as pd
gwangju = pd.read_csv("data/gwangju.csv")
gwangju_recent_4w = pd.read_csv("data/gwangju_recent_4w.csv")
gwangju.head()

,사건번호,용도,시도,시군구,소재지,감정가,최저가,결과,낙찰가,낙찰율,매각일,분기,기간구분
0,광주 4계 2020-4001,단독주택,광주,동구,광주 동구 산수동 554-31,"35,242,240","35,242,240",기각,0,0%,2026-02-19,2026_1Q,0
1,광주 6계 2023-9878,근린상가,광주,서구,"광주 서구 화정동 23-147 ,23-247 메리앙혼수백화점 2층 205호","124,000,000","124,000,000",진행,0,0%,2026-03-11,2026_1Q,0
2,광주 6계 2024-1031,근린상가,광주,남구,광주 남구 봉선동 465-3 케이씨빌딩 4층 401호,"2,759,000,000","1,545,040,000",진행,0,0%,2026-03-11,2026_1Q,0
3,광주 12계 2024-2485,답,광주,북구,광주 북구 월출동 92,"1,907,121,000","1,334,985,000",진행,0,0%,2026-03-06,2026_1Q,0
4,광주 6계 2024-5538(1),오피스텔(주거),광주,동구,"광주 동구 금남로4가 79-1 ,충장로4가 7-3,7-14 로머스파크주건축물 10층...","313,000,000","112,180,000",진행,0,0%,2026-03-11,2026_1Q,0


In [12]:
gwangju.shape, gwangju_recent_4w.shape

((55991, 13), (463, 12))

In [13]:
# (1/1) 값을 '취하'로 변경
# 괄호 제거 전이라면 '(1/1)'을, 이미 제거되어 빈 문자열이라면 ''을 변경
mask_1 = gwangju_recent_4w['결과'] == '(1/1)'
mask_2 = gwangju_recent_4w['결과'] == ''
if mask_1.any() or mask_2.any():
    gwangju_recent_4w.loc[mask_1 | mask_2, '결과'] = '취하'
    print("'(1/1)' (또는 빈 값) -> '취하' 변경 완료")
else:
    print("변경할 '(1/1)' 값이 없습니다.")


# 결과 컬럼 정리 (괄호 제거)
# 예: '낙찰(10/14)' -> '낙찰'
gwangju_recent_4w['결과'] = gwangju_recent_4w['결과'].str.split('(').str[0]

print("[결과 컬럼 정리 완료]")
print(gwangju_recent_4w['결과'].unique())

# 매각일 날짜 변환
gwangju_recent_4w['매각일'] = pd.to_datetime(gwangju_recent_4w['매각일'])

# 분기 컬럼 생성 (YYYY_nQ 형식)
gwangju_recent_4w['분기'] = gwangju_recent_4w['매각일'].dt.year.astype(str) + '_' + gwangju_recent_4w['매각일'].dt.quarter.astype(str) + 'Q'

# 소재지에서 '시도', '시군구' 추출
gwangju_recent_4w['시도'] = gwangju_recent_4w['소재지'].str.split(' ').str[0]
gwangju_recent_4w['시군구'] = gwangju_recent_4w['소재지'].str.split(' ').str[1]


print("[시도, 시군구 추출 완료]")
cols = ["사건번호", "용도", "시도", "시군구", "소재지", "감정가", "최저가", "결과", "낙찰가", "낙찰율", "매각일", "분기"]
len(cols)

gwangju_recent_4w = gwangju_recent_4w[cols]

변경할 '(1/1)' 값이 없습니다.
[결과 컬럼 정리 완료]
['기각' '낙찰' '진행' '신건' '취하' '유찰' '변경' '정지']
[시도, 시군구 추출 완료]


In [14]:
gwangju_recent_4w.to_csv('data/gwangju_recent_4w.csv', index=False, encoding='utf-8-sig')

In [15]:
from dataclasses import dataclass
from typing import List, Tuple
import pandas as pd


@dataclass
class MergeReport:
    original_rows: int
    update_rows: int
    update_rows_after_dedup: int
    duplicates_against_original: int
    new_rows_appended: int
    merged_rows: int


def _normalize_key_columns(
    df: pd.DataFrame,
    key_cols: List[str],
    strip: bool = True,
    collapse_spaces: bool = True,
) -> pd.DataFrame:
    """
    key 컬럼의 공백/표현을 정규화해서 같은 값인데 중복이 안 잡히는 문제를 줄임.
    - NaN -> ""
    - 문자열 strip, 연속공백 축소(optional)
    """
    out = df.copy()

    for c in key_cols:
        if c not in out.columns:
            raise KeyError(f"key column not found: {c}")

        s = out[c].astype("string").fillna("")
        if strip:
            s = s.str.strip()
        if collapse_spaces:
            s = s.str.replace(r"\s+", " ", regex=True)
        out[c] = s

    return out


def merge_append_new_only(
    original_df: pd.DataFrame,
    update_df: pd.DataFrame,
    key_cols: List[str],
    *,
    keep_update: str = "last",
    normalize_keys: bool = True,
    prepend: bool = True,   # ✅ True면 신규행을 맨 위로 붙임
) -> Tuple[pd.DataFrame, MergeReport]:
    """
    ✅ 중복 제거 후 '원본에 없는 row만' 원본에 합치는 함수
    - prepend=True: new_rows가 맨 위
    - prepend=False: new_rows가 맨 아래
    """
    if normalize_keys:
        o = _normalize_key_columns(original_df, key_cols)
        u = _normalize_key_columns(update_df, key_cols)
    else:
        o = original_df.copy()
        u = update_df.copy()

    # 1) update_df 내부 중복 제거
    u_dedup = u.drop_duplicates(subset=key_cols, keep=keep_update)

    # 2) 원본 key 집합
    orig_keys = set(map(tuple, o[key_cols].to_numpy()))

    # 3) 원본에 없는 key만 추출
    u_keys = list(map(tuple, u_dedup[key_cols].to_numpy()))
    mask_new = [k not in orig_keys for k in u_keys]
    new_rows = u_dedup.loc[mask_new].copy()

    # ✅ 4) 합치기: prepend 옵션에 따라 순서 결정
    if prepend:
        merged = pd.concat([new_rows, o], ignore_index=True)
    else:
        merged = pd.concat([o, new_rows], ignore_index=True)

    report = MergeReport(
        original_rows=len(original_df),
        update_rows=len(update_df),
        update_rows_after_dedup=len(u_dedup),
        duplicates_against_original=len(u_dedup) - len(new_rows),
        new_rows_appended=len(new_rows),
        merged_rows=len(merged),
    )
    return merged, report


def merge_upsert_by_key(
    original_df: pd.DataFrame,
    update_df: pd.DataFrame,
    key_cols: List[str],
    *,
    keep_update: str = "last",
    normalize_keys: bool = True,
    prepend: bool = True,  # ✅ True면 update(갱신/신규) 행을 맨 위로
) -> Tuple[pd.DataFrame, MergeReport]:
    """
    ✅ (옵션) upsert: 같은 key면 원본을 update_df 값으로 '갱신'하고
       없는 key는 append까지.

    - prepend=True: update_df(갱신/신규)가 맨 위
    - prepend=False: update_df(갱신/신규)가 맨 아래
    """
    if normalize_keys:
        o = _normalize_key_columns(original_df, key_cols)
        u = _normalize_key_columns(update_df, key_cols)
    else:
        o = original_df.copy()
        u = update_df.copy()

    u_dedup = u.drop_duplicates(subset=key_cols, keep=keep_update)

    # key tuple 생성
    o["_key__"] = list(map(tuple, o[key_cols].to_numpy()))
    u_dedup["_key__"] = list(map(tuple, u_dedup[key_cols].to_numpy()))

    update_keys = set(u_dedup["_key__"].tolist())

    # 원본에서 업데이트 대상 key 제거 → 나머지 원본 유지
    o_kept = o.loc[~o["_key__"].isin(update_keys)].copy()

    # ✅ prepend 옵션에 따라 결합 순서 결정
    if prepend:
        merged = pd.concat([u_dedup, o_kept], ignore_index=True)
    else:
        merged = pd.concat([o_kept, u_dedup], ignore_index=True)

    merged = merged.drop(columns=["_key__"], errors="ignore")

    report = MergeReport(
        original_rows=len(original_df),
        update_rows=len(update_df),
        update_rows_after_dedup=len(u_dedup),
        duplicates_against_original=len(u_dedup),
        new_rows_appended=len(u_dedup),
        merged_rows=len(merged),
    )
    return merged, report

In [16]:
key_cols = ["사건번호", "매각일", "소재지"]

merged_df, report = merge_append_new_only(gwangju, gwangju_recent_4w, key_cols)
print(report)
merged_df.shape

MergeReport(original_rows=55991, update_rows=463, update_rows_after_dedup=463, duplicates_against_original=463, new_rows_appended=0, merged_rows=55991)


(55991, 13)

In [17]:
import re
import pandas as pd

def convert_maegakil_yyyymmdd_dots_to_dashes(
    df: pd.DataFrame,
    col: str = "매각일",
    inplace: bool = False,
) -> pd.DataFrame:
    """
    매각일 컬럼이 'yyyy.mm.dd' 형태면 'yyyy-mm-dd'로 변환한다.
    - 예: '2025.02.04' -> '2025-02-04'
    - 공백 포함/앞뒤 텍스트가 섞인 경우에도 날짜 패턴만 추출해 변환 시도
    - 변환 불가/비어있음은 원값 유지

    Parameters
    ----------
    df : pandas DataFrame
    col : 대상 컬럼명 (default '매각일')
    inplace : True면 원본 df를 직접 수정

    Returns
    -------
    변환된 DataFrame
    """
    if col not in df.columns:
        raise KeyError(f"column not found: {col}")

    out = df if inplace else df.copy()

    def _convert_one(x):
        if pd.isna(x):
            return x
        s = str(x).strip()
        if not s:
            return s

        # 'yyyy.mm.dd' 패턴 찾기 (앞뒤 텍스트가 있어도 날짜만 잡음)
        m = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", s)
        if not m:
            return s

        y, mo, d = m.group(1), m.group(2).zfill(2), m.group(3).zfill(2)
        return f"{y}-{mo}-{d}"

    out[col] = out[col].apply(_convert_one)
    return out

In [18]:
merged_df
merged_df = convert_maegakil_yyyymmdd_dots_to_dashes(merged_df, col="매각일")


In [19]:
merged_df

,사건번호,용도,시도,시군구,소재지,감정가,최저가,결과,낙찰가,낙찰율,매각일,분기,기간구분
0,광주 4계 2020-4001,단독주택,광주,동구,광주 동구 산수동 554-31,"35,242,240","35,242,240",기각,0,0%,2026-02-19,2026_1Q,0.0
1,광주 6계 2023-9878,근린상가,광주,서구,"광주 서구 화정동 23-147 ,23-247 메리앙혼수백화점 2층 205호","124,000,000","124,000,000",진행,0,0%,2026-03-11,2026_1Q,0.0
2,광주 6계 2024-1031,근린상가,광주,남구,광주 남구 봉선동 465-3 케이씨빌딩 4층 401호,"2,759,000,000","1,545,040,000",진행,0,0%,2026-03-11,2026_1Q,0.0
3,광주 12계 2024-2485,답,광주,북구,광주 북구 월출동 92,"1,907,121,000","1,334,985,000",진행,0,0%,2026-03-06,2026_1Q,0.0
4,광주 6계 2024-5538(1),오피스텔(주거),광주,동구,"광주 동구 금남로4가 79-1 ,충장로4가 7-3,7-14 로머스파크주건축물 10층...","313,000,000","112,180,000",진행,0,0%,2026-03-11,2026_1Q,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
55986,광주 10계 2000-57274,차량,광주,남구,광주 남구 봉선동 무등2차아파트내 보관,"2,000,000","2,000,000",낙찰,"3,100,000",155.00%,2001-02-26,2001_1Q,3.0
55987,순천 6계 1999-51911,대지,광주,동구,광주 동구 계림동 518-45,"183,141,200","183,141,200",유찰,0,0%,2001-01-15,2001_1Q,3.0
55988,순천 6계 1999-51911,상가,광주,동구,광주 동구 계림동 518-66,"232,006,520","232,006,520",낙찰,"115,500,000",49.80%,2001-01-15,2001_1Q,3.0
55989,순천 6계 1999-54439,단독주택,광주,북구,광주 북구 양산동 284-31,"71,189,650","31,892,960",낙찰,"33,123,000",46.50%,2001-01-15,2001_1Q,3.0


In [20]:
import pandas as pd
from dateutil.relativedelta import relativedelta

# 1) 매각일을 datetime으로 변환 (문자열이면 반드시 필요)
# - errors='coerce' : 파싱 실패하면 NaT로 만들어서 터지지 않게 함
merged_df['매각일'] = pd.to_datetime(merged_df['매각일'], errors='coerce')

if not merged_df.empty:
    # 2) 기준일(가장 최근 매각일) 산출 (NaT는 자동 제외)
    last_date = merged_df['매각일'].max()

    if pd.isna(last_date):
        # 전부 NaT면 기간구분을 만들 수 없음
        merged_df['기간구분'] = pd.NA
        print("매각일이 전부 파싱 실패(NaT)라 기간구분을 생성할 수 없습니다.")
    else:
        d3  = last_date - relativedelta(months=3)
        d6  = last_date - relativedelta(months=6)
        d12 = last_date - relativedelta(months=12)

        def get_period_code(dt):
            # dt가 NaT면 구분 불가
            if pd.isna(dt):
                return pd.NA
            if dt > d3:
                return 0
            elif dt > d6:
                return 1
            elif dt > d12:
                return 2
            else:
                return 3

        merged_df['기간구분'] = merged_df['매각일'].apply(get_period_code)
        print(f"기준일: {last_date.date()} 기반 기간구분 생성 완료")

기준일: 2026-03-11 기반 기간구분 생성 완료


In [21]:
merged_df

,사건번호,용도,시도,시군구,소재지,감정가,최저가,결과,낙찰가,낙찰율,매각일,분기,기간구분
0,광주 4계 2020-4001,단독주택,광주,동구,광주 동구 산수동 554-31,"35,242,240","35,242,240",기각,0,0%,2026-02-19,2026_1Q,0
1,광주 6계 2023-9878,근린상가,광주,서구,"광주 서구 화정동 23-147 ,23-247 메리앙혼수백화점 2층 205호","124,000,000","124,000,000",진행,0,0%,2026-03-11,2026_1Q,0
2,광주 6계 2024-1031,근린상가,광주,남구,광주 남구 봉선동 465-3 케이씨빌딩 4층 401호,"2,759,000,000","1,545,040,000",진행,0,0%,2026-03-11,2026_1Q,0
3,광주 12계 2024-2485,답,광주,북구,광주 북구 월출동 92,"1,907,121,000","1,334,985,000",진행,0,0%,2026-03-06,2026_1Q,0
4,광주 6계 2024-5538(1),오피스텔(주거),광주,동구,"광주 동구 금남로4가 79-1 ,충장로4가 7-3,7-14 로머스파크주건축물 10층...","313,000,000","112,180,000",진행,0,0%,2026-03-11,2026_1Q,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
55986,광주 10계 2000-57274,차량,광주,남구,광주 남구 봉선동 무등2차아파트내 보관,"2,000,000","2,000,000",낙찰,"3,100,000",155.00%,2001-02-26,2001_1Q,3
55987,순천 6계 1999-51911,대지,광주,동구,광주 동구 계림동 518-45,"183,141,200","183,141,200",유찰,0,0%,2001-01-15,2001_1Q,3
55988,순천 6계 1999-51911,상가,광주,동구,광주 동구 계림동 518-66,"232,006,520","232,006,520",낙찰,"115,500,000",49.80%,2001-01-15,2001_1Q,3
55989,순천 6계 1999-54439,단독주택,광주,북구,광주 북구 양산동 284-31,"71,189,650","31,892,960",낙찰,"33,123,000",46.50%,2001-01-15,2001_1Q,3


In [22]:
merged_df.to_csv('data/gwangju.csv', index=False, encoding='utf-8-sig')

In [23]:
merged_df.head()

,사건번호,용도,시도,시군구,소재지,감정가,최저가,결과,낙찰가,낙찰율,매각일,분기,기간구분
0,광주 4계 2020-4001,단독주택,광주,동구,광주 동구 산수동 554-31,"35,242,240","35,242,240",기각,0,0%,2026-02-19,2026_1Q,0
1,광주 6계 2023-9878,근린상가,광주,서구,"광주 서구 화정동 23-147 ,23-247 메리앙혼수백화점 2층 205호","124,000,000","124,000,000",진행,0,0%,2026-03-11,2026_1Q,0
2,광주 6계 2024-1031,근린상가,광주,남구,광주 남구 봉선동 465-3 케이씨빌딩 4층 401호,"2,759,000,000","1,545,040,000",진행,0,0%,2026-03-11,2026_1Q,0
3,광주 12계 2024-2485,답,광주,북구,광주 북구 월출동 92,"1,907,121,000","1,334,985,000",진행,0,0%,2026-03-06,2026_1Q,0
4,광주 6계 2024-5538(1),오피스텔(주거),광주,동구,"광주 동구 금남로4가 79-1 ,충장로4가 7-3,7-14 로머스파크주건축물 10층...","313,000,000","112,180,000",진행,0,0%,2026-03-11,2026_1Q,0
